In [ ]:
# ==========================================
# TAHAP 1: PRA-PROCESSING DATA & EDA (KLASIFIKASI BINER)
# ==========================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import RobustScaler

# Set representasi visual di notebook
%matplotlib inline
sns.set_theme(style="whitegrid")

# 1. Membaca Dataset
# Ganti 'nama_file_dataset_kamu.csv' dengan nama file CSV asli Anda
df = pd.read_csv('climate.csv') 
print("--- Dimensi Data Awal ---")
print(f"Jumlah Baris: {df.shape[0]}, Jumlah Kolom: {df.shape[1]}\n")

# 2. Menghapus Kolom yang Tidak Digunakan
# Menghapus kolom ID stasiun dan arah angin mentah (karena bertipe derajat/kategorikal)
kolom_dihapus = ['station_id', 'ddd_x', 'ddd_car']
df = df.drop(columns=kolom_dihapus, errors='ignore')
print(f"Kolom setelah pembersihan awal: {df.columns.tolist()}\n")

# 3. Feature Engineering Kolom Date & Pembuatan Target Biner Besok (Klasifikasi)
# Menambahkan dayfirst=True agar Pandas tidak error membaca format tanggal Indonesia (Hari-Bulan-Tahun)
df['date'] = pd.to_datetime(df['date'], dayfirst=True)
df = df.sort_values(by='date').reset_index(drop=True) # Urutkan berdasarkan waktu kronologis

# Ekstrak fitur musiman numerik yang sangat penting untuk menangkap pola iklim
df['month'] = df['date'].dt.month
df['dayofyear'] = df['date'].dt.dayofyear

# Membuat target besok (menggeser data RR maju 1 hari)
df['RR_tomorrow'] = df['RR'].shift(-1)

# Transformasi Target menjadi Biner sesuai standar BMKG: 
# Curah hujan >= 0.5 mm = Hujan (1), di bawah 0.5 mm = Tidak Hujan (0)
df['Target_Besok'] = (df['RR_tomorrow'] >= 0.5).astype(int)

# Hapus kolom tanggal asli dan kolom regresi lama agar tidak terjadi kebocoran data (data leakage)
df = df.drop(columns=['date', 'RR_tomorrow'])

# Hapus baris paling terakhir karena data besoknya otomatis kosong (NaN) setelah di-shift
df = df.dropna(subset=['Target_Besok']) 
print("Feature Engineering selesai. Fitur musiman dan target biner 'Target_Besok' berhasil dibuat.\n")


# 4. Menampilkan & Handling Missing Value
print("--- Jumlah Missing Values Per Kolom ---")
print(df.isnull().sum())
print("\n")

# Skenario BMKG: Ganti nilai anomali 8888 (hujan tak terukur) dengan 0, dan 9999 dengan NaN
df['RR'] = df['RR'].replace(8888, 0)
df = df.replace(9999, np.nan)

# Imputasi missing values numerik menggunakan Median (Sangat aman untuk distribusi data cuaca)
for col in df.columns:
    if col != 'Target_Besok':
        df[col] = df[col].fillna(df[col].median())
print("Handling Missing Value Selesai (Semua data kosong telah diimputasi dengan median).\n")

print("--- Jumlah Missing Values Per Kolom ---")
print(df.isnull().sum())
print("\n")


# 5. Menampilkan & Menghapus Data Duplicate
duplicate_count = df.duplicated().sum()
print(f"Jumlah data duplikat ditemukan: {duplicate_count}")
df[df.duplicated()]

if duplicate_count > 0:
    # Menghapus duplikat, mempertahankan baris pertama yang muncul
    df = df.drop_duplicates(keep='first')
    print("Data duplikat telah berhasil dihapus.\n")
    
    
 # 6. Menampilkan Boxplot Per Variabel Secara Terpisah (Lebih Besar & Jelas)
features_to_check = ['Tn', 'Tx', 'Tavg', 'RH_avg', 'RR', 'ss', 'ff_x', 'ff_avg']
fig, axes = plt.subplots(nrows=4, ncols=2, figsize=(16, 18))
axes = axes.flatten()

for i, col in enumerate(features_to_check):
    # Menggunakan .values untuk menghindari error kecocokan versi Pandas/Seaborn
    sns.boxplot(y=df[col].values, ax=axes[i], color='skyblue', width=0.4)
    axes[i].set_title(f'Deteksi Outliers - Kolom {col}', fontsize=12, fontweight='bold')
    axes[i].set_ylabel('Nilai / Satuan', fontsize=10)
    axes[i].set_xlabel(col, fontsize=10)

plt.tight_layout()
plt.show()

# *Catatan Penting: Langkah menghapus outliers dilewati (dipertahankan) karena ini data cuaca ekstrem nyata.*

# 7. Memvisualisasikan Distribusi Masing-Masing Variabel
df.hist(figsize=(15, 12), bins=20, edgecolor='black')
plt.suptitle("Distribusi Masing-Masing Variabel", y=0.92, fontsize=16)
plt.show()

# 8. Menampilkan Deskripsi Statistik Data
print("--- Deskripsi Statistik Dataset ---")
print(df.describe().T)
print("\n")

# 9. Grafik, Tabel, dan Urutan Korelasi
# Membuat matriks korelasi
corr_matrix = df.corr()

# Menampilkan Grafik Korelasi (Heatmap)
plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title("Matriks Korelasi Antar Variabel")
plt.show()

# Menampilkan Tabel Nilai Korelasi (Matriks mentah)
print("--- Tabel Matriks Korelasi ---")
print(corr_matrix)
print("\n")

# Mengurutkan Nilai Korelasi terhadap variabel target klasifikasi 'Target_Besok'
print("--- Urutan Korelasi Variabel terhadap Target Biner (Target_Besok) ---")
print(corr_matrix['Target_Besok'].sort_values(ascending=False))
print("\n")

# 10. Tampilan Distribusi Kelas Target (Imbalance Check)
print("--- Distribusi Kelas Target Besok (0: Tidak Hujan, 1: Hujan) ---")
print(df['Target_Besok'].value_counts())
print(df['Target_Besok'].value_counts(normalize=True) * 100)
print("\n")

# 11. Menscaling Data (RobustScaler - Aman untuk Data yang Memiliki Outliers)
# Pisahkan variabel independen (X) dan target (y)
X = df.drop(columns=['Target_Besok'])
y = df['Target_Besok'].values

# Menggunakan RobustScaler karena kita berkomitmen mempertahankan pencilan cuaca ekstrem
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)
print("Data variabel independen berhasil discaling menggunakan RobustScaler.\n")


# 12. Menampilkan Scree Plot (Analisis PCA untuk Reduksi Dimensi Visual)
from sklearn.decomposition import PCA
pca = PCA()
pca.fit(X_scaled)
explained_variance = pca.explained_variance_ratio_

plt.figure(figsize=(8, 4))
plt.bar(range(1, len(explained_variance) + 1), explained_variance, alpha=0.7, align='center', label='Individual variance')
plt.step(range(1, len(explained_variance) + 1), np.cumsum(explained_variance), where='mid', label='Cumulative variance', color='red')
plt.ylabel('Explained Variance Ratio')
plt.xlabel('Principal Components')
plt.title('Scree Plot (PCA Analysis)')
plt.legend(loc='best')
plt.tight_layout()
plt.show()

# 13. Menyimpan Data Hasil Pra-processing ke CSV khusus Klasifikasi
# Menggabungkan kembali fitur hasil scaling dengan target biner
df_final = X_scaled_df.copy()
df_final['Target_Besok'] = y

df_final.to_csv('dataset_rain_classification.csv', index=False)
print("--- TAHAP 1 SELESAI 100% ---")
print("Data bersih siap pakai untuk klasifikasi telah disimpan dengan nama: 'dataset_rain_classification.csv'")

In [ ]:
# ==========================================
# TAHAP 2: PEMODELAN, TUNING, & EVALUASI
# ==========================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

# Sklearn & Pemodelan
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Metrik Evaluasi
from sklearn.metrics import (
    accuracy_score, classification_report, 
    roc_curve, roc_auc_score
)


# 1. Membaca Data Bersih Hasil Tahap 1 (Sebelum di-scaling massal)
# Pastikan Anda memuat file CSV yang fiturnya masih dalam skala asli, bukan hasil scaler luar.
df_processed = pd.read_csv('dataset_rain_classification.csv')

# 2. Menampilkan Variabel Independen dan Dependen
X = df_processed.drop(columns=['Target_Besok'])
y = df_processed['Target_Besok']

print("--- Verifikasi Kolom ---")
print(f"Variabel Independen (X): {X.columns.tolist()}")
print(f"Variabel Dependen (y): {['Target_Besok']}\n")

# 3. Pemisahan Data Train dan Test dari Data MENTAH (Mencegah Data Leakage)
# Menggunakan stratify=y agar proporsi kelas target (Hujan/Tidak) seimbang di data train & test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("--- Hasil Pemisahan Data (Baris, Kolom) ---")
print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"y_train: {y_train.shape} | y_test: {y_test.shape}\n")

print("--- Verifikasi Proporsi Kelas Target (%) ---")
print(f"Proporsi di Data Train:\n{y_train.value_counts(normalize=True) * 100}")
print(f"Proporsi di Data Test:\n{y_test.value_counts(normalize=True) * 100}\n")

# 4. Pemodelan Awal (Baseline Model menggunakan Pipeline)
# Menyiapkan Scaler DI DALAM pipeline agar fitting hanya terjadi pada data Train
models = {
    'Logistic Regression': LogisticRegression(max_iter=100, random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss')
}

baseline_results = {}

print("--- Menjalankan Baseline Model ---")
for name, model in models.items():
    # Definisikan Pipeline: Scaler dimasukkan di sini
    pipeline = Pipeline([
        ('scaler', RobustScaler()),
        ('classifier', model)
    ])
    
    # Fitting pipeline ke data training
    pipeline.fit(X_train, y_train)
    
    # Prediksi
    y_pred = pipeline.predict(X_test)
    y_pred_proba = pipeline.predict_proba(X_test)[:, 1]
    
    # Hitung Skor
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_pred_proba)
    baseline_results[name] = {'Accuracy': acc, 'AUC-ROC': auc}
    
    print(f"\n[Baseline] {name} Performance:")
    print(f"Accuracy: {acc:.4f} | AUC-ROC: {auc:.4f}")
    print(classification_report(y_test, y_pred))

    
# 5. Hyperparameter Tuning Menggunakan RandomizedSearchCV (Lebih Cepat)
from sklearn.model_selection import RandomizedSearchCV

print("\n--- Memulai Proses Hyperparameter Tuning ---")

# Grid parameter telah disederhanakan dan disesuaikan untuk klasifikasi biner
param_grids = {
    'Logistic Regression': {
        'classifier__C': [0.01, 0.1, 1.0, 10.0],  # C jauh lebih penting daripada max_iter
        'classifier__max_iter': [100, 200]
    },
    'Random Forest': {
        'classifier__n_estimators': [10, 50, 100, 200, 500],  # Menghapus 500 karena terlalu lama
        'classifier__max_depth': [10,20],       # Menghapus None untuk menghindari overfitting
        'classifier__min_samples_split': [2, 5]      # Parameter alternatif yang berdampak besar
    },
    'XGBoost': {
        'classifier__n_estimators': [10, 50, 100, 200, 500],
        'classifier__max_depth': [5, 10],          # Kedalaman 10+ atau None sangat memperlambat XGBoost
        'classifier__learning_rate': [0.05, 0.1, 0.2]
    }
}

best_pipelines = {}
tuned_results = {}

for name, model in models.items():
    print(f"Tuning {name}...")
    pipeline = Pipeline([
        ('scaler', RobustScaler()),
        ('classifier', model)
    ])
    
    # Menggunakan RandomizedSearchCV dengan n_iter=10 untuk membatasi jumlah kombinasi
    # n_jobs=-1 tetap digunakan untuk memanfaatkan semua core CPU Anda
    random_search = RandomizedSearchCV(
        pipeline, 
        param_distributions=param_grids[name], 
        n_iter=10, 
        cv=3, 
        scoring='roc_auc', 
        n_jobs=-1, 
        random_state=42
    )
    
    random_search.fit(X_train, y_train)
    
    best_pipelines[name] = random_search.best_estimator_
    
    # Evaluasi Hasil Tuning
    y_pred = random_search.predict(X_test)
    y_pred_proba = random_search.predict_proba(X_test)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_pred_proba)
    tuned_results[name] = {'Accuracy': acc, 'AUC-ROC': auc}
    
    print(f"Selesai tuning {name}. Best Score (AUC): {random_search.best_score_:.4f}")
    
# 6. Pembuatan Summary DataFrame
summary_df = pd.DataFrame({
    'Baseline Accuracy': [baseline_results[m]['Accuracy'] for m in models],
    'Tuned Accuracy': [tuned_results[m]['Accuracy'] for m in models],
    'Baseline AUC-ROC': [baseline_results[m]['AUC-ROC'] for m in models],
    'Tuned AUC-ROC': [tuned_results[m]['AUC-ROC'] for m in models]
}, index=models.keys())

print("\n--- Summary Performa Model ---")
print(summary_df)
print("\n")

In [ ]:
%%writefile app.py
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import streamlit as st
from PIL import Image


# =========================================================
# 1. KONFIGURASI DASHBOARD
# =========================================================

st.set_page_config(
    page_title="Rain Tomorrow Prediction",
    page_icon="🌧️",
    layout="wide",
    initial_sidebar_state="expanded"
)

MODEL_PATH = Path("best_rain_prediction_model.pkl")
IMAGE_PATH = Path("sunflower_rain.jpg")


# =========================================================
# 2. HEADER DASHBOARD
# =========================================================

def show_header():
    # Membagi halaman menjadi bagian teks dan gambar
    col_text, col_image = st.columns(
        [3, 2],
        gap="large",
        vertical_alignment="center"
    )

    with col_text:
        st.title("Welcome to my first machine learning dashboard")

        st.subheader(
            "This dashboard created by Jesi Pebralia"
        )

        st.write(
            "This application predicts whether it will rain "
            "tomorrow based on today's weather observations."
        )

        st.caption(
            "Machine Learning | Weather Forecasting | Indonesia Climate"
        )

    with col_image:
        if IMAGE_PATH.exists():
            st.image(
                Image.open(IMAGE_PATH),
                use_container_width=True
            )
        else:
            st.warning(
                "Gambar tidak ditemukan. Simpan gambar sebagai "
                "sunflower_rain.jpg di folder aplikasi."
            )

    st.divider()


# =========================================================
# 3. MEMUAT MODEL PICKLE
# =========================================================

@st.cache_resource
def load_model():
    with open(MODEL_PATH, "rb") as file:
        model_export = pickle.load(file)

    pipeline = model_export["pipeline"]
    threshold = float(model_export["best_threshold"])
    features = model_export["features"]

    return pipeline, threshold, features


# =========================================================
# 4. INFORMASI APLIKASI
# =========================================================

def rain():
    st.header("🌧️ Rain Tomorrow Prediction")

    st.write(
        """
        Aplikasi ini digunakan untuk memprediksi kondisi hujan
        atau tidak hujan pada hari berikutnya berdasarkan data
        meteorologi hari ini.

        Pengguna dapat memasukkan data cuaca secara manual
        melalui slider atau mengunggah file CSV. Model machine
        learning yang telah dilatih sebelumnya kemudian digunakan
        untuk menghasilkan prediksi.

        Dataset:
        [Indonesia Climate - Kaggle]
        (https://www.kaggle.com/datasets/greegtitan/indonesia-climate)
        """
    )


# =========================================================
# 5. INPUT DATA MENGGUNAKAN SLIDER
# =========================================================

def manual_input():
    st.subheader("Input Data Cuaca Hari Ini")

    data = {}

    col1, col2 = st.columns(2)

    with col1:
        st.markdown("#### Suhu dan Kelembapan")

        data["Tn"] = st.slider(
            "Suhu minimum (°C)",
            -10.0, 35.0, 23.0, 0.1
        )

        data["Tx"] = st.slider(
            "Suhu maksimum (°C)",
            -5.0, 45.0, 32.0, 0.1
        )

        data["Tavg"] = st.slider(
            "Suhu rata-rata (°C)",
            -10.0, 40.0, 27.0, 0.1
        )

        data["RH_avg"] = st.slider(
            "Kelembapan rata-rata (%)",
            0.0, 100.0, 80.0, 1.0
        )

        data["RR"] = st.slider(
            "Curah hujan hari ini (mm)",
            0.0, 300.0, 0.0, 0.1
        )

    with col2:
        st.markdown("#### Penyinaran dan Angin")

        data["ss"] = st.slider(
            "Durasi penyinaran matahari (jam)",
            0.0, 12.0, 6.0, 0.1
        )

        data["ff_x"] = st.slider(
            "Kecepatan angin maksimum (m/s)",
            0.0, 40.0, 5.0, 0.1
        )

        data["ff_avg"] = st.slider(
            "Kecepatan angin rata-rata (m/s)",
            0.0, 30.0, 2.0, 0.1
        )

        data["month"] = st.slider(
            "Bulan dalam setahun",
            1, 12, 1, 1
        )

        data["dayofyear"] = st.slider(
            "Hari dalam setahun",
            1, 366, 1, 1
        )

    return pd.DataFrame([data])


# =========================================================
# 6. FUNGSI PREDIKSI
# =========================================================

def predict_rain(data, pipeline, threshold, features):
    missing = [
        feature for feature in features
        if feature not in data.columns
    ]

    if missing:
        raise ValueError(
            "Kolom fitur tidak lengkap: "
            + ", ".join(missing)
        )

    X = data[features].copy()

    # Pastikan semua fitur numerik
    for column in features:
        X[column] = pd.to_numeric(
            X[column], errors="raise"
        )

    if X.isnull().any().any():
        raise ValueError(
            "Data mengandung nilai kosong (NaN)."
        )

    if not np.isfinite(X.to_numpy(dtype=float)).all():
        raise ValueError(
            "Data mengandung nilai yang tidak valid."
        )

    if hasattr(pipeline, "predict_proba"):
        probabilities = pipeline.predict_proba(X)

        # Asumsi: kelas positif adalah kelas hujan.
        # Pastikan urutan kelas sesuai dengan target training.
        positive_probability = probabilities[:, 1]

        predictions = (
            positive_probability >= threshold
        ).astype(int)

        return predictions, positive_probability

    predictions = pipeline.predict(X)

    return predictions, None


# =========================================================
# 7. MENAMPILKAN HASIL PREDIKSI
# =========================================================

def show_results(data, pipeline, threshold, features):
    try:
        predictions, probabilities = predict_rain(
            data,
            pipeline,
            threshold,
            features
        )

        st.subheader("Hasil Prediksi untuk Besok")

        if probabilities is not None:
            # Hasil ini mengasumsikan label 1 = hujan
            # dan label 0 = tidak hujan.
            for i, prediction in enumerate(predictions):
                if prediction == 1:
                    st.error(
                        f"🌧️ Data ke-{i + 1}: "
                        "Diprediksi HUJAN besok"
                    )
                elif prediction == 0:
                    st.success(
                        f"☀️ Data ke-{i + 1}: "
                        "Diprediksi TIDAK HUJAN besok"
                    )
                else:
                    st.warning(
                        f"Label prediksi tidak dikenali: {prediction}"
                    )

                st.metric(
                    f"Probabilitas hujan — data ke-{i + 1}",
                    f"{probabilities[i] * 100:.2f}%"
                )

            st.caption(
                f"Threshold model: {threshold:.3f}"
            )

            results = data.copy()
            results["Prediksi_kode"] = predictions
            results["Prediksi"] = [
                "Hujan" if p == 1 else "Tidak Hujan"
                for p in predictions
            ]
            results["Probabilitas_hujan_persen"] = (
                probabilities * 100
            ).round(2)

        else:
            results = data.copy()
            results["Prediksi_kode"] = predictions

            st.info(
                "Model tidak menyediakan probabilitas. "
                "Hasil berikut berasal dari predict()."
            )

        st.dataframe(
            results,
            use_container_width=True
        )

        st.download_button(
            label="Unduh Hasil Prediksi (CSV)",
            data=results.to_csv(index=False).encode("utf-8"),
            file_name="rain_prediction_results.csv",
            mime="text/csv"
        )

    except (ValueError, KeyError) as error:
        st.error(f"Prediksi gagal: {error}")


# =========================================================
# 8. FUNGSI UTAMA APLIKASI
# =========================================================

def main():

    # PENTING: kedua tulisan pembuka ditampilkan paling atas
    show_header()

    # Informasi aplikasi prediksi
    rain()

    st.divider()

    # Memuat model tersimpan
    try:
        pipeline, threshold, features = load_model()

    except FileNotFoundError:
        st.error(
            "File model tidak ditemukan. Pastikan "
            "best_rain_prediction_model.pkl berada "
            "di folder yang sama dengan app.py."
        )
        st.stop()

    except (KeyError, pickle.UnpicklingError, EOFError) as error:
        st.error(f"Model gagal dimuat: {error}")
        st.stop()

    # Pilihan metode input
    st.sidebar.title("🌦️ Menu Prediksi")

    input_method = st.sidebar.radio(
        "Pilih metode input data:",
        [
            "Input Manual (Slider)",
            "Upload File CSV"
        ]
    )

    # -----------------------------------------------------
    # INPUT MANUAL
    # -----------------------------------------------------

    if input_method == "Input Manual (Slider)":

        input_data = manual_input()

        if st.button(
            "🔮 Prediksi Hujan Besok",
            type="primary"
        ):
            show_results(
                input_data,
                pipeline,
                threshold,
                features
            )

    # -----------------------------------------------------
    # INPUT CSV
    # -----------------------------------------------------

    else:
        st.subheader("Upload File CSV")

        uploaded_file = st.file_uploader(
            "Pilih file CSV berisi data cuaca hari ini",
            type=["csv"]
        )

        if uploaded_file is not None:
            try:
                input_data = pd.read_csv(uploaded_file)

                st.write("Pratinjau data:")
                st.dataframe(
                    input_data.head(),
                    use_container_width=True
                )

                missing = [
                    f for f in features
                    if f not in input_data.columns
                ]

                if missing:
                    st.error(
                        "Kolom berikut belum tersedia: "
                        + ", ".join(missing)
                    )

                elif st.button(
                    "🔮 Prediksi Semua Data CSV",
                    type="primary"
                ):
                    show_results(
                        input_data,
                        pipeline,
                        threshold,
                        features
                    )

            except (
                pd.errors.ParserError,
                UnicodeDecodeError
            ) as error:
                st.error(
                    f"File CSV tidak dapat dibaca: {error}"
                )

    # Footer
    st.divider()
    st.caption(
        "Rain Tomorrow Prediction | Created by Jesi Pebralia"
    )


# =========================================================
# 9. MENJALANKAN APLIKASI
# =========================================================

if __name__ == "__main__":
    main()
